In [2]:
import pandas as pd
df = pd.read_csv("spambase.data", header=None)
df.shape

(4601, 58)

In [3]:
lines = open("spambase.names").read().splitlines()
names = [l.split(":")[0] for l in lines if l.endswith("continuous.") and not l.startswith("|")]
print(len(names))
print(names)

57
['word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d', 'word_freq_our', 'word_freq_over', 'word_freq_remove', 'word_freq_internet', 'word_freq_order', 'word_freq_mail', 'word_freq_receive', 'word_freq_will', 'word_freq_people', 'word_freq_report', 'word_freq_addresses', 'word_freq_free', 'word_freq_business', 'word_freq_email', 'word_freq_you', 'word_freq_credit', 'word_freq_your', 'word_freq_font', 'word_freq_000', 'word_freq_money', 'word_freq_hp', 'word_freq_hpl', 'word_freq_george', 'word_freq_650', 'word_freq_lab', 'word_freq_labs', 'word_freq_telnet', 'word_freq_857', 'word_freq_data', 'word_freq_415', 'word_freq_85', 'word_freq_technology', 'word_freq_1999', 'word_freq_parts', 'word_freq_pm', 'word_freq_direct', 'word_freq_cs', 'word_freq_meeting', 'word_freq_original', 'word_freq_project', 'word_freq_re', 'word_freq_edu', 'word_freq_table', 'word_freq_conference', 'char_freq_;', 'char_freq_(', 'char_freq_[', 'char_freq_!', 'char_freq_$', 'char_freq_#', 'cap

In [4]:
df.columns = names + ["is_spam"]
df.head()

,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,char_freq_;,char_freq_(,char_freq_[,char_freq_!,char_freq_$,char_freq_#,capital_run_length_average,capital_run_length_longest,capital_run_length_total,is_spam
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278,1
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,1
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,1


In [5]:
print(df.shape)
print("Nulls:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
print(df["is_spam"].value_counts())

(4601, 58)
Nulls: 0
Duplicates: 391
is_spam
0    2788
1    1813
Name: count, dtype: int64


In [6]:
df = df.drop_duplicates().reset_index(drop=True)
X = df.drop("is_spam", axis=1)
y = df["is_spam"]
print(X.shape, y.shape)

(4210, 57) (4210,)


In [7]:
%pip install "scipy<1.14"

Note: you may need to restart the kernel to use updated packages.


In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)
print(y_test.value_counts())

(3368, 57) (842, 57)
is_spam
0    506
1    336
Name: count, dtype: int64


In [17]:
test_df = X_test.copy()
test_df["is_spam"] = y_test
test_df.to_csv("test_data.csv", index=False)
print(test_df.shape)

(842, 58)


In [21]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Naive Bayes": GaussianNB(),
    "SVM": make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained")

Logistic Regression trained
Decision Tree trained
KNN trained
Naive Bayes trained


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM trained


In [35]:
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef)
rows = []
for name, model in models.items():
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    rows.append({
         "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, pred),
        "AUC": roc_auc_score(y_test, prob),
        "precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "MCC": matthews_corrcoef(y_test, pred),
    })
results = pd.DataFrame(rows).round(4)
results

,ML Model Name,Accuracy,AUC,precision,Recall,F1,MCC
0,Logistic Regression,0.9347,0.9768,0.9297,0.9048,0.9170,0.8634
1,Decision Tree,0.9074,0.9053,0.8817,0.8869,0.8843,0.8071
2,KNN,0.9133,0.9546,0.9148,0.8631,0.8882,0.8184
3,Naive Bayes,0.8361,0.9525,0.7240,0.9524,0.8226,0.6975
4,SVM,0.9323,0.9772,0.9319,0.8958,0.9135,0.8584


In [37]:
import joblib, os
os.makedirs("model", exist_ok=True)
for name, m in models.items():
    joblib.dump(m, "model/" + name.lower().replace(" ", "_") + ".joblib")

print(os.listdir("model"))

['svm.joblib', 'naive_bayes.joblib', 'logistic_regression.joblib', 'decision_tree.joblib', 'knn.joblib']


In [39]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

st.set_page_config(page_title="Spam Detector", layout="wide")
st.title("Spam Email Classifier")
st.write("Upload test data, pick a model, and see how it performs.")

MODEL_FILES = {
    "Logistic Regression": "model/logistic_regression.joblib",
    "Decision Tree": "model/decision_tree.joblib",
    "KNN": "model/knn.joblib",
    "Naive Bayes": "model/naive_bayes.joblib",
    "SVM": "model/svm.joblib",
}

uploaded = st.file_uploader("Upload test CSV", type="csv")
choice = st.selectbox("Choose a model", list(MODEL_FILES.keys()))

if uploaded is not None:
    data = pd.read_csv(uploaded)
    st.subheader("Data preview")
    st.dataframe(data.head())

Writing app.py


In [41]:
%%writefile -a app.py

    TARGET = "is_spam"
    if TARGET not in data.columns:
        st.error("The CSV must contain the 'is_spam' target column.")
    else:
        from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                                     recall_score, f1_score, matthews_corrcoef,
                                     confusion_matrix, classification_report)
        import matplotlib.pyplot as plt
        import seaborn as sns

        X_new = data.drop(columns=[TARGET])
        y_new = data[TARGET]
        model = joblib.load(MODEL_FILES[choice])
        pred = model.predict(X_new)
        prob = model.predict_proba(X_new)[:, 1]

        st.subheader("Evaluation metrics: " + choice)
        c1, c2, c3 = st.columns(3)
        c1.metric("Accuracy", round(accuracy_score(y_new, pred), 4))
        c2.metric("AUC", round(roc_auc_score(y_new, prob), 4))
        c3.metric("MCC", round(matthews_corrcoef(y_new, pred), 4))
        c4, c5, c6 = st.columns(3)
        c4.metric("Precision", round(precision_score(y_new, pred), 4))
        c5.metric("Recall", round(recall_score(y_new, pred), 4))
        c6.metric("F1", round(f1_score(y_new, pred), 4))

        st.subheader("Confusion matrix")
        fig, ax = plt.subplots(figsize=(4, 3))
        sns.heatmap(confusion_matrix(y_new, pred), annot=True, fmt="d",
                    cmap="Blues", ax=ax)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        st.pyplot(fig)

        st.subheader("Classification report")
        st.text(classification_report(y_new, pred))

Appending to app.py


In [43]:
import sklearn

reqs = [
    "streamlit",
    "scikit-learn==" + sklearn.__version__,
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "joblib",
]
open("requirements.txt", "w").write("\n".join(reqs) + "\n")
print(open("requirements.txt").read())

streamlit
scikit-learn==1.9.1
numpy
pandas
matplotlib
seaborn
joblib

